In [ ]:
from pathlib import Path

import teehr
from teehr import RemoteReadWriteEvaluation
from teehr.utilities.apply_migrations import evolve_catalog_schema

from setup_utils import DEV_LOCATION_ID_LIST
from teehr.evaluation.spark_session_utils import create_spark_session

#### Start the spark session

In [ ]:
spark = create_spark_session()

In [ ]:
migrations_dir = Path(teehr.__file__).parent / "migrations"

evolve_catalog_schema(
    spark=spark,
    migrations_dir_path=migrations_dir,
    target_catalog_name="iceberg",
    target_namespace_name="teehr"
)

#### Initialize a TEEHR Evaluation with read/write privileges

In [ ]:
ev = RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

#### Now you can pull in a subset of data from the TEEHR-Cloud warehouse via the API
Note: If you encounter a time out error, try reducing the page_size arg

Download domain variables, location data, and attribute data

In [ ]:
print("Loading configurations.")
ev.download.configurations(load=True)

print("Loading units.")
ev.download.units(load=True)

print("Loading variables.")
ev.download.variables(load=True)

print("Loading locations.")
ev.download.locations(
    ids=DEV_LOCATION_ID_LIST,
    load=True
)
print("Loading location croswalks.")
ev.download.location_crosswalks(
    primary_location_id=DEV_LOCATION_ID_LIST,
    load=True
)
print("Loading location attributes.")
ev.download.attributes(load=True)
ev.download.location_attributes(
    location_id=DEV_LOCATION_ID_LIST,
    load=True
)
print("Loading domain variables, locations, crosswalks, and attributes complete!")

Download forecast timeseries and obs

In [ ]:
ev.download.primary_timeseries(
    primary_location_id=DEV_LOCATION_ID_LIST,
    configuration_name="usgs_observations",
    variable_name="streamflow_none_inst",
    start_date="2026-05-01",
    end_date="2026-05-12",
    load=True,
)

ev.download.secondary_timeseries(
    primary_location_id=DEV_LOCATION_ID_LIST,
    configuration_name="nwm30_medium_range",
    variable_name="streamflow_hourly_inst",
    reference_time="2026-05-01/2026-05-02",
    load=True,
)

ev.download.secondary_timeseries(
    primary_location_id=DEV_LOCATION_ID_LIST,
    configuration_name="nwm30_short_range_hawaii",
    variable_name="streamflow_15min_inst",
    reference_time="2026-05-01/2026-05-10",
    load=True,
)

ev.download.secondary_timeseries(
    primary_location_id=DEV_LOCATION_ID_LIST,
    configuration_name="nwm30_short_range_alaska",
    variable_name="streamflow_hourly_inst",
    reference_time="2026-05-01/2026-05-10",
    load=True,
)

ev.download.secondary_timeseries(
    primary_location_id=DEV_LOCATION_ID_LIST,
    configuration_name="nwm30_short_range_puertorico",
    variable_name="streamflow_hourly_inst",
    reference_time="2026-05-01/2026-05-10",
    load=True,
)
print("Loading forecast timeseries complete!")

Download retrospective simulation timeseries and obs

In [ ]:
ev.download.primary_timeseries(
    primary_location_id=DEV_LOCATION_ID_LIST,
    configuration_name="usgs_observations",
    variable_name="streamflow_none_inst",
    start_date="2020-05-01",
    end_date="2020-05-31",
    load=True,
)

ev.download.secondary_timeseries(
    primary_location_id=DEV_LOCATION_ID_LIST,
    configuration_name="nwm30_retrospective",
    variable_name="streamflow_hourly_inst",
    start_date="2020-05-01",
    end_date="2020-05-31",
    load=True,
)
print("Loading retrospective timeseries complete!")

Now download HUC6 and USGS basins for the data management and gridded dashboards

In [ ]:
ev.download.locations(prefix="huc6", load=True)

In [ ]:
basin_ids = [gage_id.replace("usgs-", "usgsbasin-") for gage_id in DEV_LOCATION_ID_LIST]
ev.download.locations(ids=basin_ids, load=True)

To enable the data management dashboards run the `update-data-management-tables`
prefect workflow. It builds the two tables the dashboards read:
`configurations_by_location` and `configurations_summary`.

For the retrospective dashboard run the following notebooks:
- `02_create_joined_timeseries.ipynb`
- `03_generate_basic_metrics notebooks.ipynb`

For the forecast dashboard run the following prefect workflows:
- `update-joined-forecast-table`
- `update-forecast-metrics-table`

To load vector tiles for the gridded dashboard run the `create-vector-tile-archive` prefect workflow specifying either the "usgsbasin" or "huc6" prefix, and updating the output layer name accordingly (this defines its name in the dashboard).

For gridded mean areal values on the USGS basins loaded above, run the following prefect workflows:
- `ingest-nwm31-forcing-analysis-assim-icechunk`
- `calculate-nwm31-forcing-analysis-assim-icechunk-pixel-coverage-weights`
- `calculate-nwm31-forcing-analysis-assim-icechunk-mean-areal-values`

In [ ]:
spark.stop()